In [3]:
from dotenv import load_dotenv
load_dotenv()

True

In [14]:
import sys
import os

# Ensure Python can find your 'modules' folder
sys.path.append(os.path.abspath('..'))

from modules.ground_truth import generate_ground_truth_dataset

# Define where your files live
input_path = 'data/knowledge-base.json' # Change this if your file is somewhere else!
output_path = 'data/ground_truth.json'

# Let it run! (This might take a minute depending on how big your JSON is)
generate_ground_truth_dataset(input_filepath=input_path, output_filepath=output_path)

Generating questions for 115 documents...


  0%|          | 0/115 [00:00<?, ?it/s]


Done! Saved 115 records to data/ground_truth.json


In [15]:
import os
import json
import pandas as pd
from tqdm.auto import tqdm
from openai import OpenAI

# Import your custom modules
import sys
sys.path.append(os.path.abspath('..')) # Ensures it can find 'modules' if running in notebooks/
from modules.ingest import load_faq_data, build_indices
from modules.rag_helper import RAGBase, VectorRAG

# 1. Load Data & Build Both Retrievers
# Note: Adjust the path inside load_faq_data() if it can't find knowledge-base.json
documents = load_faq_data() 
keyword_index, vector_index = build_indices(documents)

client = OpenAI()
keyword_rag = RAGBase(index=keyword_index, llm_client=client)
vector_rag = VectorRAG(index=vector_index, llm_client=client)

# 2. Load Your Newly Generated Ground Truth Dataset
# This path check ensures it works whether you run it from the root or the notebooks folder
gt_path = '../data/ground_truth.json' if os.path.exists('../data/ground_truth.json') else 'data/ground_truth.json'

with open(gt_path, 'r') as f:
    ground_truth = json.load(f)

print(f"Loaded {len(ground_truth)} test questions from ground truth.")

# 3. Dedicated MRR & Hit Rate Evaluation Function
def calculate_mrr_and_hit_rate(rag_system, dataset, num_results=5):
    total_queries = len(dataset)
    hits = 0
    reciprocal_ranks = []

    for item in tqdm(dataset, desc="Evaluating Retriever"):
        question = item['user_query']
        expected_chunk = item['chunk_id']
        
        # Perform retrieval using just the search method
        results = rag_system.search(question, num_results=num_results)
        
        # Find position of expected chunk
        found_rank = 0
        for rank, doc in enumerate(results, start=1):
            if doc.get('chunk_id') == expected_chunk:
                found_rank = rank
                break
        
        # Calculate Hits and MRR math
        if found_rank > 0:
            hits += 1
            reciprocal_ranks.append(1.0 / found_rank)
        else:
            reciprocal_ranks.append(0.0)

    hit_rate = hits / total_queries
    mrr = sum(reciprocal_ranks) / total_queries
    
    return hit_rate, mrr

# 4. Run Evaluation on Keyword Search
print("\n--- Evaluating Keyword Search (minsearch) ---")
kw_hit_rate, kw_mrr = calculate_mrr_and_hit_rate(keyword_rag, ground_truth)

# 5. Run Evaluation on Vector Search
print("\n--- Evaluating Vector Search (FastEmbed) ---")
vec_hit_rate, vec_mrr = calculate_mrr_and_hit_rate(vector_rag, ground_truth)

# 6. Summary Comparison Table
results_df = pd.DataFrame([
    {
        "Retriever": "Keyword Search (minsearch)", 
        "Hit Rate (@5)": f"{kw_hit_rate * 100:.2f}%", 
        "MRR": f"{kw_mrr:.4f}"
    },
    {
        "Retriever": "Vector Search (FastEmbed)", 
        "Hit Rate (@5)": f"{vec_hit_rate * 100:.2f}%", 
        "MRR": f"{vec_mrr:.4f}"
    }
])

print("\n=== RETRIEVAL EVALUATION SUMMARY ===")
print(results_df.to_string(index=False))

Building Keyword Index...
Building Vector Index...
Both indices built successfully!
Loaded 115 test questions from ground truth.

--- Evaluating Keyword Search (minsearch) ---


Evaluating Retriever:   0%|          | 0/115 [00:00<?, ?it/s]


--- Evaluating Vector Search (FastEmbed) ---


Evaluating Retriever:   0%|          | 0/115 [00:00<?, ?it/s]


=== RETRIEVAL EVALUATION SUMMARY ===
                 Retriever Hit Rate (@5)    MRR
Keyword Search (minsearch)        60.00% 0.3752
 Vector Search (FastEmbed)        71.30% 0.4277
